[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/image-denoising.ipynb)

# Image Denoising: From Traditional Methods to Deep Learning

In this comprehensive notebook, we'll explore the fascinating world of **image denoising** - the process of removing noise from corrupted images to recover clean, high-quality versions.

## Learning Objectives

By the end of this notebook, you will:

1. **Understand different types of image noise** and their characteristics
2. **Learn to add synthetic noise** to clean images for training and evaluation
3. **Implement traditional denoising methods** as baselines
4. **Build a Denoising Autoencoder (DAE)** from scratch
5. **Implement U-Net architecture** with skip connections
6. **Compare different approaches** quantitatively and qualitatively
7. **Evaluate using standard metrics** (PSNR, SSIM)
8. **Understand blind denoising** challenges

## Why Image Denoising?

Image denoising is a fundamental problem in computer vision with numerous real-world applications:

- **Medical Imaging**: Enhancing MRI, CT scans, and X-rays for better diagnosis
- **Photography**: Improving low-light photos from smartphones and cameras
- **Astronomy**: Cleaning telescope images to reveal distant celestial objects
- **Surveillance**: Enhancing security camera footage in poor conditions
- **Scientific Imaging**: Improving electron microscopy and other scientific instruments

## Notebook Structure

We'll progress through the following parts:

**Part 1: Understanding Image Noise** (Theory)
- Types of noise and their sources
- Visualizing different noise patterns

**Part 2: Adding Synthetic Noise** (Implementation)
- Gaussian noise
- Salt-and-pepper noise
- Speckle noise

**Part 3: Traditional Denoising Methods** (Baseline)
- Gaussian blur
- Median filter
- Bilateral filter

**Part 4: Denoising Autoencoder (DAE)** (Deep Learning)
- Architecture design
- Training procedure
- Evaluation

**Part 5: U-Net Architecture** (Advanced)
- Skip connections
- Multi-scale features
- Superior performance

**Part 6: Comparison and Analysis** (Evaluation)
- Quantitative metrics
- Visual comparisons
- Blind denoising

**Part 7: Advanced Topics** (Extensions)
- Varying noise levels
- Real-world considerations
- Future directions

Let's begin our journey into image denoising!

## Setup and Imports

First, let's import all necessary libraries and set up our environment.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import math
from tqdm.auto import tqdm
from typing import Tuple, Optional, List

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Configure device (MPS for macOS, CUDA for Linux/Windows, CPU fallback)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Metal Performance Shaders) device")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA device: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using CPU device")

# Configure matplotlib for better visualizations
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print(f"PyTorch version: {torch.__version__}")
print("Setup complete!")

---

# Part 1: Understanding Image Noise

## What is Image Noise?

**Image noise** refers to random variations in brightness or color information in images. It's an undesirable byproduct of image capture and transmission that degrades image quality.

### Sources of Noise

Noise can arise from various sources:

1. **Sensor Noise**: Electronic circuits in camera sensors generate random fluctuations
2. **Photon Noise**: Quantum nature of light causes statistical variations
3. **Low Light**: Insufficient light leads to higher relative noise
4. **High ISO**: Amplifying weak signals also amplifies noise
5. **Transmission Errors**: Corruption during data transfer
6. **Compression Artifacts**: Lossy compression can introduce noise-like artifacts

## Mathematical Model

We can model a noisy image as:

$$y = x + n$$

Where:
- $y$ is the observed noisy image
- $x$ is the clean (true) image we want to recover
- $n$ is the noise component

The goal of **denoising** is to recover $x$ from $y$:

$$\hat{x} = f(y)$$

Where $f$ is our denoising function (which could be a traditional algorithm or a neural network), and $\hat{x}$ is our estimate of the clean image.

## Types of Noise

Let's explore three common types of noise:

### 1. Gaussian Noise

**Gaussian noise** (also called **normal noise**) follows a normal distribution:

$$n \sim \mathcal{N}(\mu, \sigma^2)$$

- $\mu$ is the mean (usually 0)
- $\sigma^2$ is the variance (controls noise strength)

**Characteristics:**
- Most common in electronic sensors
- Additive noise (independent of signal)
- Relatively smooth and continuous

### 2. Salt-and-Pepper Noise

**Salt-and-pepper noise** (also called **impulse noise**) randomly replaces pixels with minimum (pepper) or maximum (salt) values:

$$y_i = \begin{cases} 
0 & \text{with probability } p_{pepper} \\
255 & \text{with probability } p_{salt} \\
x_i & \text{otherwise}
\end{cases}$$

**Characteristics:**
- Occurs due to transmission errors or dead pixels
- Sparse (affects only some pixels)
- High contrast (black/white spots)

### 3. Speckle Noise

**Speckle noise** is multiplicative noise common in coherent imaging:

$$y = x \cdot (1 + n)$$

Where $n$ is typically Gaussian.

**Characteristics:**
- Common in SAR, ultrasound, and laser imaging
- Multiplicative (scales with signal strength)
- Granular texture appearance

Let's visualize these noise types!

---

# Part 2: Adding Synthetic Noise

To train and evaluate denoising models, we need to create noisy versions of clean images. Let's implement functions to add different types of noise.

## Theory: Why Synthetic Noise?

For supervised learning, we need **paired data**: (noisy image, clean image). In practice:

1. **Clean images are easy to obtain** (any high-quality image dataset)
2. **We can synthetically add noise** to create training pairs
3. **This gives us perfect ground truth** for training

The assumption is that if a model learns to remove synthetic noise, it will generalize to real-world noise (which often has similar statistical properties).

In [ ]:
def add_gaussian_noise(images: torch.Tensor, noise_std: float = 0.1) -> torch.Tensor:
    """
    Add Gaussian noise to images.
    
    Args:
        images: Clean images, shape (B, C, H, W), values in [0, 1]
        noise_std: Standard deviation of Gaussian noise
    
    Returns:
        Noisy images, clipped to [0, 1]
    """
    noise = torch.randn_like(images) * noise_std
    noisy = images + noise
    return torch.clamp(noisy, 0, 1)


def add_salt_pepper_noise(images: torch.Tensor, salt_prob: float = 0.05, 
                          pepper_prob: float = 0.05) -> torch.Tensor:
    """
    Add salt-and-pepper noise to images.
    
    Args:
        images: Clean images, shape (B, C, H, W), values in [0, 1]
        salt_prob: Probability of salt (white) noise
        pepper_prob: Probability of pepper (black) noise
    
    Returns:
        Noisy images with salt-and-pepper noise
    """
    noisy = images.clone()
    
    # Add salt (white pixels)
    salt_mask = torch.rand_like(images) < salt_prob
    noisy[salt_mask] = 1.0
    
    # Add pepper (black pixels)
    pepper_mask = torch.rand_like(images) < pepper_prob
    noisy[pepper_mask] = 0.0
    
    return noisy


def add_speckle_noise(images: torch.Tensor, noise_std: float = 0.1) -> torch.Tensor:
    """
    Add speckle (multiplicative) noise to images.
    
    Args:
        images: Clean images, shape (B, C, H, W), values in [0, 1]
        noise_std: Standard deviation of multiplicative noise
    
    Returns:
        Noisy images with speckle noise, clipped to [0, 1]
    """
    noise = torch.randn_like(images) * noise_std
    noisy = images * (1 + noise)
    return torch.clamp(noisy, 0, 1)


print("Noise functions implemented!")

## Load Sample Images

Let's load some clean images from MNIST to visualize the different noise types.

In [ ]:
# Load MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
])

mnist_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

# Get a batch of clean images
sample_loader = DataLoader(mnist_dataset, batch_size=8, shuffle=True)
clean_images, labels = next(iter(sample_loader))

print(f"Loaded {len(clean_images)} sample images")
print(f"Image shape: {clean_images.shape}")  # (B, 1, 28, 28)
print(f"Value range: [{clean_images.min():.3f}, {clean_images.max():.3f}]")

## Visualize Different Noise Types

Now let's apply each noise type to our sample images and visualize the results.

In [ ]:
def show_image_grid(images: torch.Tensor, title: str, nrow: int = 8):
    """Display a grid of images."""
    grid = make_grid(images, nrow=nrow, normalize=False, pad_value=0.5)
    plt.figure(figsize=(12, 3))
    plt.imshow(grid.permute(1, 2, 0).cpu(), cmap='gray' if images.shape[1] == 1 else None)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()


# Create noisy versions
gaussian_noisy = add_gaussian_noise(clean_images, noise_std=0.2)
salt_pepper_noisy = add_salt_pepper_noise(clean_images, salt_prob=0.05, pepper_prob=0.05)
speckle_noisy = add_speckle_noise(clean_images, noise_std=0.3)

# Visualize
show_image_grid(clean_images, "Clean Images (Ground Truth)")
show_image_grid(gaussian_noisy, "Gaussian Noise (σ=0.2)")
show_image_grid(salt_pepper_noisy, "Salt-and-Pepper Noise (p=0.05)")
show_image_grid(speckle_noisy, "Speckle Noise (σ=0.3)")

## Detailed Comparison

Let's examine a single image with different noise types side-by-side.

In [ ]:
# Select one image for detailed comparison
idx = 0
clean_img = clean_images[idx:idx+1]

# Apply different noise levels
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Comparison of Noise Types on a Single Image', fontsize=16, fontweight='bold')

# Row 1: Different noise types
axes[0, 0].imshow(clean_img[0, 0].cpu(), cmap='gray')
axes[0, 0].set_title('Clean Image')
axes[0, 0].axis('off')

axes[0, 1].imshow(add_gaussian_noise(clean_img, 0.1)[0, 0].cpu(), cmap='gray')
axes[0, 1].set_title('Gaussian (σ=0.1)')
axes[0, 1].axis('off')

axes[0, 2].imshow(add_gaussian_noise(clean_img, 0.2)[0, 0].cpu(), cmap='gray')
axes[0, 2].set_title('Gaussian (σ=0.2)')
axes[0, 2].axis('off')

axes[0, 3].imshow(add_gaussian_noise(clean_img, 0.3)[0, 0].cpu(), cmap='gray')
axes[0, 3].set_title('Gaussian (σ=0.3)')
axes[0, 3].axis('off')

# Row 2: More noise types
axes[1, 0].imshow(add_salt_pepper_noise(clean_img, 0.02, 0.02)[0, 0].cpu(), cmap='gray')
axes[1, 0].set_title('Salt-Pepper (p=0.02)')
axes[1, 0].axis('off')

axes[1, 1].imshow(add_salt_pepper_noise(clean_img, 0.05, 0.05)[0, 0].cpu(), cmap='gray')
axes[1, 1].set_title('Salt-Pepper (p=0.05)')
axes[1, 1].axis('off')

axes[1, 2].imshow(add_speckle_noise(clean_img, 0.2)[0, 0].cpu(), cmap='gray')
axes[1, 2].set_title('Speckle (σ=0.2)')
axes[1, 2].axis('off')

axes[1, 3].imshow(add_speckle_noise(clean_img, 0.4)[0, 0].cpu(), cmap='gray')
axes[1, 3].set_title('Speckle (σ=0.4)')
axes[1, 3].axis('off')

plt.tight_layout()
plt.show()

print("\nObservations:")
print("- Gaussian noise: Smooth, affects all pixels uniformly")
print("- Salt-and-pepper: Sparse, high-contrast black/white spots")
print("- Speckle: Granular texture, stronger in brighter regions")

### 🤔 Reflection Questions

1. **Which noise type is hardest to see in dark regions of the image? Why?**
   - Hint: Consider how each noise type scales with pixel intensity

2. **If you were designing a camera sensor, which noise type would be most concerning?**
   - Hint: Think about which is most common and hardest to remove

3. **How might the choice of noise type affect our training strategy?**
   - Hint: Consider whether different noise types require different denoising approaches

---

# Part 3: Traditional Denoising Methods (Baselines)

Before diving into deep learning, let's implement and understand traditional denoising methods. These will serve as our **baselines** for comparison.

## Theory: Classical Approaches

Traditional methods rely on assumptions about image structure and noise properties:

### 1. Gaussian Blur

**Idea**: Average each pixel with its neighbors using a Gaussian-weighted kernel.

**Mathematical formulation**:

$$\hat{x}(i,j) = \sum_{k,l} G(k,l) \cdot y(i+k, j+l)$$

Where $G$ is a Gaussian kernel:

$$G(k,l) = \frac{1}{2\pi\sigma^2} e^{-\frac{k^2+l^2}{2\sigma^2}}$$

**Pros**: Simple, fast, smooths noise
**Cons**: Blurs edges and fine details

### 2. Median Filter

**Idea**: Replace each pixel with the median of its neighbors.

$$\hat{x}(i,j) = \text{median}\{y(i+k, j+l) : (k,l) \in \mathcal{N}\}$$

Where $\mathcal{N}$ is a neighborhood (e.g., 3×3 or 5×5 window).

**Pros**: Excellent for salt-and-pepper noise, preserves edges
**Cons**: Computationally expensive, can remove fine details

### 3. Bilateral Filter

**Idea**: Weighted average that considers both spatial distance and intensity similarity.

$$\hat{x}(i,j) = \frac{1}{W} \sum_{k,l} G_s(k,l) \cdot G_r(y(i,j), y(i+k,j+l)) \cdot y(i+k,j+l)$$

Where:
- $G_s$ is spatial Gaussian (distance-based weight)
- $G_r$ is range Gaussian (intensity-based weight)
- $W$ is normalization factor

**Pros**: Preserves edges while smoothing, very effective
**Cons**: Slow, parameters need tuning

Let's implement these methods!

In [ ]:
def gaussian_blur_denoise(images: torch.Tensor, kernel_size: int = 5, 
                          sigma: float = 1.0) -> torch.Tensor:
    """
    Denoise images using Gaussian blur.
    
    Args:
        images: Noisy images, shape (B, C, H, W)
        kernel_size: Size of Gaussian kernel (odd number)
        sigma: Standard deviation of Gaussian kernel
    
    Returns:
        Denoised images
    """
    channels = images.shape[1]
    
    # Create Gaussian kernel
    kernel = torch.zeros((kernel_size, kernel_size))
    center = kernel_size // 2
    
    for i in range(kernel_size):
        for j in range(kernel_size):
            x, y = i - center, j - center
            kernel[i, j] = math.exp(-(x**2 + y**2) / (2 * sigma**2))
    
    kernel = kernel / kernel.sum()  # Normalize
    kernel = kernel.view(1, 1, kernel_size, kernel_size).repeat(channels, 1, 1, 1)
    kernel = kernel.to(images.device)
    
    # Apply convolution
    padding = kernel_size // 2
    denoised = F.conv2d(images, kernel, padding=padding, groups=channels)
    
    return denoised


def median_filter_denoise(images: torch.Tensor, kernel_size: int = 3) -> torch.Tensor:
    """
    Denoise images using median filter.
    
    Args:
        images: Noisy images, shape (B, C, H, W)
        kernel_size: Size of median filter window
    
    Returns:
        Denoised images
    """
    B, C, H, W = images.shape
    padding = kernel_size // 2
    
    # Pad images
    padded = F.pad(images, (padding, padding, padding, padding), mode='reflect')
    
    # Unfold to extract patches
    patches = F.unfold(padded, kernel_size=kernel_size)
    patches = patches.view(B, C, kernel_size * kernel_size, H * W)
    
    # Compute median for each patch
    denoised = patches.median(dim=2)[0]
    denoised = denoised.view(B, C, H, W)
    
    return denoised


print("Traditional denoising methods implemented!")

## Apply Traditional Methods

Let's apply these methods to our noisy images and compare the results.

In [ ]:
# Create noisy test images
test_clean = clean_images[:4]
test_noisy_gaussian = add_gaussian_noise(test_clean, noise_std=0.2)
test_noisy_salt_pepper = add_salt_pepper_noise(test_clean, salt_prob=0.05, pepper_prob=0.05)

# Apply traditional methods
gaussian_denoised_g = gaussian_blur_denoise(test_noisy_gaussian, kernel_size=5, sigma=1.0)
median_denoised_g = median_filter_denoise(test_noisy_gaussian, kernel_size=3)

gaussian_denoised_sp = gaussian_blur_denoise(test_noisy_salt_pepper, kernel_size=5, sigma=1.0)
median_denoised_sp = median_filter_denoise(test_noisy_salt_pepper, kernel_size=3)

print("Denoising complete! Visualizing results...")

In [ ]:
# Visualize results for Gaussian noise
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.suptitle('Traditional Methods on Gaussian Noise', fontsize=16, fontweight='bold')

for i in range(4):
    axes[i, 0].imshow(test_clean[i, 0].cpu(), cmap='gray')
    axes[i, 0].set_title('Clean' if i == 0 else '')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(test_noisy_gaussian[i, 0].cpu(), cmap='gray')
    axes[i, 1].set_title('Noisy (σ=0.2)' if i == 0 else '')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(gaussian_denoised_g[i, 0].cpu(), cmap='gray')
    axes[i, 2].set_title('Gaussian Blur' if i == 0 else '')
    axes[i, 2].axis('off')
    
    axes[i, 3].imshow(median_denoised_g[i, 0].cpu(), cmap='gray')
    axes[i, 3].set_title('Median Filter' if i == 0 else '')
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Visualize results for salt-and-pepper noise
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.suptitle('Traditional Methods on Salt-and-Pepper Noise', fontsize=16, fontweight='bold')

for i in range(4):
    axes[i, 0].imshow(test_clean[i, 0].cpu(), cmap='gray')
    axes[i, 0].set_title('Clean' if i == 0 else '')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(test_noisy_salt_pepper[i, 0].cpu(), cmap='gray')
    axes[i, 1].set_title('Noisy (p=0.05)' if i == 0 else '')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(gaussian_denoised_sp[i, 0].cpu(), cmap='gray')
    axes[i, 2].set_title('Gaussian Blur' if i == 0 else '')
    axes[i, 2].axis('off')
    
    axes[i, 3].imshow(median_denoised_sp[i, 0].cpu(), cmap='gray')
    axes[i, 3].set_title('Median Filter' if i == 0 else '')
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()

## Key Observations

From the visualizations above:

### Gaussian Noise:
- **Gaussian blur**: Reduces noise effectively but also blurs edges and details
- **Median filter**: Preserves edges better but may not remove all noise

### Salt-and-Pepper Noise:
- **Gaussian blur**: Smears the black/white spots, creating gray halos
- **Median filter**: Excellent performance! Removes most impulse noise while preserving edges

**Lesson**: Different noise types require different denoising strategies. Traditional methods make assumptions about noise that may not hold in all cases.

### 🤔 Reflection Questions

1. **Why does the median filter work so well for salt-and-pepper noise?**
   - Hint: Think about how outliers affect the mean vs. median

2. **What happens to fine details (like edges and textures) with Gaussian blur?**
   - Hint: Look closely at the digit boundaries

3. **Can you think of scenarios where traditional methods would fail completely?**
   - Hint: Consider very heavy noise or complex noise patterns

---

# Part 4: Denoising Autoencoder (DAE)

Now let's move to **deep learning** approaches! We'll start with a **Denoising Autoencoder (DAE)**.

## Theory: Autoencoders for Denoising

### What is an Autoencoder?

An **autoencoder** is a neural network that learns to compress (encode) and reconstruct (decode) data:

```
Input → Encoder → Latent Code → Decoder → Output
```

Mathematically:

$$z = f_{enc}(x) \quad \text{(encoder)}$$
$$\hat{x} = f_{dec}(z) \quad \text{(decoder)}$$

We train to minimize reconstruction loss: $\mathcal{L} = ||x - \hat{x}||^2$

### Denoising Autoencoder

For **denoising**, we make a simple but powerful modification:

1. **Input**: Noisy image $y = x + n$
2. **Target**: Clean image $x$
3. **Loss**: $\mathcal{L} = ||x - f(y)||^2$

The network learns to map noisy images to clean ones!

### Architecture Design

For images, we use **convolutional layers**:

**Encoder** (downsampling):
```
Conv → ReLU → MaxPool → Conv → ReLU → MaxPool → ...
```
- Progressively reduces spatial dimensions
- Increases channel dimensions
- Extracts hierarchical features

**Decoder** (upsampling):
```
ConvTranspose → ReLU → ConvTranspose → ReLU → ...
```
- Progressively increases spatial dimensions
- Reduces channel dimensions
- Reconstructs the image

**Final layer**: Often uses Sigmoid activation to ensure output is in [0, 1]

Let's implement this!

In [ ]:
class DenoisingAutoencoder(nn.Module):
    """
    Simple Denoising Autoencoder with convolutional layers.
    
    Architecture:
    - Encoder: 3 conv blocks with downsampling
    - Decoder: 3 transposed conv blocks with upsampling
    """
    
    def __init__(self, in_channels: int = 1):
        super().__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            # Block 1: 1 -> 32 channels, 28x28 -> 14x14
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Block 2: 32 -> 64 channels, 14x14 -> 7x7
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Block 3: 64 -> 128 channels, 7x7 (no pooling)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            # Block 1: 128 -> 64 channels, 7x7 -> 14x14
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(inplace=True),
            
            # Block 2: 64 -> 32 channels, 14x14 -> 28x28
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(inplace=True),
            
            # Block 3: 32 -> 1 channel, 28x28 (final output)
            nn.Conv2d(32, in_channels, kernel_size=3, padding=1),
            nn.Sigmoid(),  # Output in [0, 1]
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Encode
        z = self.encoder(x)
        
        # Decode
        x_hat = self.decoder(z)
        
        return x_hat


# Create model and print architecture
dae = DenoisingAutoencoder(in_channels=1).to(device)
print(dae)
print(f"\nTotal parameters: {sum(p.numel() for p in dae.parameters()):,}")

## Create Training Dataset

We need a dataset that returns pairs of (noisy, clean) images.

In [ ]:
class NoisyDataset(Dataset):
    """
    Dataset wrapper that adds noise to clean images on-the-fly.
    """
    
    def __init__(self, clean_dataset: Dataset, noise_type: str = 'gaussian',
                 noise_std: float = 0.2, salt_prob: float = 0.05, pepper_prob: float = 0.05):
        self.clean_dataset = clean_dataset
        self.noise_type = noise_type
        self.noise_std = noise_std
        self.salt_prob = salt_prob
        self.pepper_prob = pepper_prob
    
    def __len__(self) -> int:
        return len(self.clean_dataset)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        clean_img, label = self.clean_dataset[idx]
        
        # Add noise
        if self.noise_type == 'gaussian':
            noisy_img = add_gaussian_noise(clean_img.unsqueeze(0), self.noise_std).squeeze(0)
        elif self.noise_type == 'salt_pepper':
            noisy_img = add_salt_pepper_noise(clean_img.unsqueeze(0), 
                                              self.salt_prob, self.pepper_prob).squeeze(0)
        elif self.noise_type == 'speckle':
            noisy_img = add_speckle_noise(clean_img.unsqueeze(0), self.noise_std).squeeze(0)
        else:
            raise ValueError(f"Unknown noise type: {self.noise_type}")
        
        return noisy_img, clean_img


# Create noisy training dataset
noisy_train_dataset = NoisyDataset(mnist_dataset, noise_type='gaussian', noise_std=0.2)

# Split into train and validation
train_size = int(0.9 * len(noisy_train_dataset))
val_size = len(noisy_train_dataset) - train_size
train_dataset, val_dataset = random_split(noisy_train_dataset, [train_size, val_size])

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")
print(f"Batches per epoch: {len(train_loader)}")

## Training Loop

Now let's train our denoising autoencoder!

In [ ]:
def train_epoch(model: nn.Module, train_loader: DataLoader, 
                optimizer: optim.Optimizer, device: torch.device) -> float:
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    
    for noisy, clean in tqdm(train_loader, desc="Training", leave=False):
        noisy, clean = noisy.to(device), clean.to(device)
        
        # Forward pass
        denoised = model(noisy)
        loss = F.mse_loss(denoised, clean)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)


def validate(model: nn.Module, val_loader: DataLoader, device: torch.device) -> float:
    """Validate the model."""
    model.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for noisy, clean in val_loader:
            noisy, clean = noisy.to(device), clean.to(device)
            denoised = model(noisy)
            loss = F.mse_loss(denoised, clean)
            total_loss += loss.item()
    
    return total_loss / len(val_loader)


# Training configuration
num_epochs = 10
learning_rate = 1e-3

optimizer = optim.Adam(dae.parameters(), lr=learning_rate)

# Training loop
train_losses = []
val_losses = []

print("Starting training...\n")
for epoch in range(num_epochs):
    train_loss = train_epoch(dae, train_loader, optimizer, device)
    val_loss = validate(dae, val_loader, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.6f}, Val Loss = {val_loss:.6f}")

print("\nTraining complete!")

## Visualize Training Progress

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Val Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Denoising Autoencoder Training Progress', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final train loss: {train_losses[-1]:.6f}")
print(f"Final val loss: {val_losses[-1]:.6f}")

## Test the Denoising Autoencoder

Let's see how well our trained model denoises images!

In [ ]:
# Get test samples
dae.eval()
test_noisy, test_clean_targets = next(iter(val_loader))
test_noisy = test_noisy[:8].to(device)
test_clean_targets = test_clean_targets[:8]

with torch.no_grad():
    test_denoised_dae = dae(test_noisy).cpu()

test_noisy = test_noisy.cpu()

# Visualize
fig, axes = plt.subplots(3, 8, figsize=(16, 6))
fig.suptitle('Denoising Autoencoder Results', fontsize=16, fontweight='bold')

for i in range(8):
    axes[0, i].imshow(test_clean_targets[i, 0], cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Clean', fontweight='bold')
    
    axes[1, i].imshow(test_noisy[i, 0], cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Noisy', fontweight='bold')
    
    axes[2, i].imshow(test_denoised_dae[i, 0], cmap='gray')
    axes[2, i].axis('off')
    if i == 0:
        axes[2, i].set_title('DAE Output', fontweight='bold')

plt.tight_layout()
plt.show()

### 🤔 Reflection Questions

1. **How does the DAE compare to traditional methods?**
   - Look at edge preservation and overall smoothness

2. **Are there any artifacts or issues in the denoised images?**
   - Check for blurriness, missing details, or strange patterns

3. **What might limit the DAE's performance?**
   - Think about the bottleneck in the architecture

---

# Part 5: U-Net Architecture

The **U-Net** is a powerful architecture that addresses the main limitation of simple autoencoders: **information loss** during downsampling.

## Theory: Skip Connections

### The Problem with Simple Autoencoders

In our DAE:
```
28×28 → 14×14 → 7×7 → 14×14 → 28×28
```

**Issue**: Fine details are lost at the bottleneck (7×7) and can't be perfectly reconstructed!

### U-Net Solution: Skip Connections

**U-Net** adds **skip connections** that bypass the bottleneck:

```
Encoder:   28×28 ──┐  14×14 ──┐  7×7
                    │          │   │
                    │          │   │
Decoder:   28×28 ←─┘  14×14 ←─┘  7×7
```

**Key idea**: Concatenate encoder features directly to decoder at each level!

$$x_{dec}^{(i)} = \text{Upsample}(x_{dec}^{(i-1)}) \oplus x_{enc}^{(i)}$$

Where $\oplus$ denotes concatenation along the channel dimension.

### Benefits

1. **Preserves fine details**: High-resolution features skip the bottleneck
2. **Multi-scale fusion**: Combines low-level (edges) and high-level (semantics) features
3. **Better gradients**: Shorter paths for gradient flow during backprop
4. **State-of-the-art**: U-Net is the go-to architecture for image-to-image tasks

### Architecture Diagram

```
Input (1, 28, 28)
      |
   Conv+ReLU → (32, 28, 28) ────────┐
      |                              |
   MaxPool → (32, 14, 14)            |
      |                              |
   Conv+ReLU → (64, 14, 14) ────┐   |
      |                          |   |
   MaxPool → (64, 7, 7)          |   |
      |                          |   |
   Conv+ReLU → (128, 7, 7)       |   |
      |                          |   |
   ConvTranspose → (64, 14, 14)  |   |
      |                          |   |
   Concat ←──────────────────────┘   |
      |                              |
   Conv+ReLU → (64, 14, 14)          |
      |                              |
   ConvTranspose → (32, 28, 28)      |
      |                              |
   Concat ←──────────────────────────┘
      |
   Conv+Sigmoid → (1, 28, 28)
      |
   Output
```

Let's implement U-Net!

In [ ]:
class UNet(nn.Module):
    """
    U-Net architecture for image denoising.
    
    Features:
    - Symmetric encoder-decoder with skip connections
    - Preserves fine details through concatenation
    - Multi-scale feature fusion
    """
    
    def __init__(self, in_channels: int = 1):
        super().__init__()
        
        # Encoder (downsampling path)
        self.enc1 = self._make_encoder_block(in_channels, 32)
        self.enc2 = self._make_encoder_block(32, 64)
        
        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        
        # Decoder (upsampling path)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = self._make_decoder_block(128, 64)  # 128 = 64 (up) + 64 (skip)
        
        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec2 = self._make_decoder_block(64, 32)  # 64 = 32 (up) + 32 (skip)
        
        # Final output layer
        self.output = nn.Sequential(
            nn.Conv2d(32, in_channels, kernel_size=1),
            nn.Sigmoid(),
        )
        
        self.pool = nn.MaxPool2d(2, 2)
    
    def _make_encoder_block(self, in_channels: int, out_channels: int) -> nn.Module:
        """Create an encoder block with two conv layers."""
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
    
    def _make_decoder_block(self, in_channels: int, out_channels: int) -> nn.Module:
        """Create a decoder block with two conv layers."""
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Encoder path
        enc1 = self.enc1(x)          # (B, 32, 28, 28)
        x = self.pool(enc1)           # (B, 32, 14, 14)
        
        enc2 = self.enc2(x)           # (B, 64, 14, 14)
        x = self.pool(enc2)           # (B, 64, 7, 7)
        
        # Bottleneck
        x = self.bottleneck(x)        # (B, 128, 7, 7)
        
        # Decoder path with skip connections
        x = self.up1(x)               # (B, 64, 14, 14)
        x = torch.cat([x, enc2], dim=1)  # (B, 128, 14, 14) - concatenate skip
        x = self.dec1(x)              # (B, 64, 14, 14)
        
        x = self.up2(x)               # (B, 32, 28, 28)
        x = torch.cat([x, enc1], dim=1)  # (B, 64, 28, 28) - concatenate skip
        x = self.dec2(x)              # (B, 32, 28, 28)
        
        # Output
        x = self.output(x)            # (B, 1, 28, 28)
        
        return x


# Create U-Net model
unet = UNet(in_channels=1).to(device)
print(unet)
print(f"\nTotal parameters: {sum(p.numel() for p in unet.parameters()):,}")

## Train U-Net

Let's train the U-Net with the same setup as our DAE for fair comparison.

In [ ]:
# Training configuration
num_epochs = 10
learning_rate = 1e-3

optimizer_unet = optim.Adam(unet.parameters(), lr=learning_rate)

# Training loop
train_losses_unet = []
val_losses_unet = []

print("Starting U-Net training...\n")
for epoch in range(num_epochs):
    train_loss = train_epoch(unet, train_loader, optimizer_unet, device)
    val_loss = validate(unet, val_loader, device)
    
    train_losses_unet.append(train_loss)
    val_losses_unet.append(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.6f}, Val Loss = {val_loss:.6f}")

print("\nU-Net training complete!")

## Compare Training Curves

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='DAE', marker='o', alpha=0.7)
plt.plot(train_losses_unet, label='U-Net', marker='s', alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Loss Comparison', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(val_losses, label='DAE', marker='o', alpha=0.7)
plt.plot(val_losses_unet, label='U-Net', marker='s', alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Validation Loss Comparison', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nFinal Validation Loss:")
print(f"DAE:   {val_losses[-1]:.6f}")
print(f"U-Net: {val_losses_unet[-1]:.6f}")
print(f"\nImprovement: {(1 - val_losses_unet[-1]/val_losses[-1])*100:.1f}%")

## Visual Comparison: DAE vs U-Net

Now let's see the visual difference between the two architectures.

In [ ]:
# Get test samples
unet.eval()
dae.eval()

test_noisy, test_clean_targets = next(iter(val_loader))
test_noisy = test_noisy[:8].to(device)
test_clean_targets = test_clean_targets[:8]

with torch.no_grad():
    test_denoised_dae = dae(test_noisy).cpu()
    test_denoised_unet = unet(test_noisy).cpu()

test_noisy = test_noisy.cpu()

# Visualize comparison
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle('Denoising Methods Comparison', fontsize=16, fontweight='bold')

for i in range(8):
    axes[0, i].imshow(test_clean_targets[i, 0], cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Clean', fontweight='bold', rotation=0, labelpad=40)
    
    axes[1, i].imshow(test_noisy[i, 0], cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Noisy', fontweight='bold', rotation=0, labelpad=40)
    
    axes[2, i].imshow(test_denoised_dae[i, 0], cmap='gray')
    axes[2, i].axis('off')
    if i == 0:
        axes[2, i].set_ylabel('DAE', fontweight='bold', rotation=0, labelpad=40)
    
    axes[3, i].imshow(test_denoised_unet[i, 0], cmap='gray')
    axes[3, i].axis('off')
    if i == 0:
        axes[3, i].set_ylabel('U-Net', fontweight='bold', rotation=0, labelpad=40)

plt.tight_layout()
plt.show()

print("\nKey observations:")
print("- U-Net preserves finer details better than DAE")
print("- U-Net edges are sharper and more accurate")
print("- Skip connections prevent information loss")

### 🤔 Reflection Questions

1. **Why does U-Net outperform the simple DAE?**
   - Think about what information is preserved through skip connections

2. **Look at the edges of digits - which model preserves them better?**
   - Compare sharpness and accuracy of boundaries

3. **What's the trade-off of using U-Net?**
   - Consider model size, training time, and complexity

---

# Part 6: Quantitative Evaluation Metrics

Visual comparison is subjective. Let's use **quantitative metrics** to objectively measure denoising quality.

## Theory: Image Quality Metrics

### 1. Peak Signal-to-Noise Ratio (PSNR)

**PSNR** measures the ratio between maximum signal power and noise power:

$$\text{PSNR} = 10 \log_{10}\left(\frac{\text{MAX}^2}{\text{MSE}}\right) = 20 \log_{10}\left(\frac{\text{MAX}}{\sqrt{\text{MSE}}}\right)$$

Where:
- $\text{MAX}$ is the maximum possible pixel value (1.0 for our normalized images)
- $\text{MSE}$ is the mean squared error between clean and denoised images

**Interpretation**:
- Higher is better (more signal, less noise)
- Typical range: 20-50 dB
- 30-40 dB is considered good quality
- Measured in **decibels (dB)**

**Pros**: Simple, fast, widely used
**Cons**: Doesn't always match human perception

### 2. Structural Similarity Index (SSIM)

**SSIM** measures structural similarity considering luminance, contrast, and structure:

$$\text{SSIM}(x, y) = \frac{(2\mu_x\mu_y + c_1)(2\sigma_{xy} + c_2)}{(\mu_x^2 + \mu_y^2 + c_1)(\sigma_x^2 + \sigma_y^2 + c_2)}$$

Where:
- $\mu_x, \mu_y$ are means
- $\sigma_x^2, \sigma_y^2$ are variances
- $\sigma_{xy}$ is covariance
- $c_1, c_2$ are stability constants

**Interpretation**:
- Range: [-1, 1], but typically [0, 1] for similar images
- 1 means perfect similarity
- 0.95-0.99 is excellent quality
- Better matches human perception than PSNR

**Pros**: Perceptually motivated, considers structure
**Cons**: More complex, slower to compute

Let's implement these metrics!

In [ ]:
def calculate_psnr(clean: torch.Tensor, denoised: torch.Tensor, max_val: float = 1.0) -> float:
    """
    Calculate Peak Signal-to-Noise Ratio (PSNR).
    
    Args:
        clean: Ground truth images
        denoised: Denoised images
        max_val: Maximum possible pixel value
    
    Returns:
        PSNR in decibels (dB)
    """
    mse = F.mse_loss(denoised, clean)
    if mse == 0:
        return float('inf')
    psnr = 20 * torch.log10(torch.tensor(max_val) / torch.sqrt(mse))
    return psnr.item()


def calculate_ssim(clean: torch.Tensor, denoised: torch.Tensor, 
                   window_size: int = 11, max_val: float = 1.0) -> float:
    """
    Calculate Structural Similarity Index (SSIM).
    
    Simplified implementation for single-channel images.
    
    Args:
        clean: Ground truth images, shape (B, 1, H, W)
        denoised: Denoised images, shape (B, 1, H, W)
        window_size: Size of Gaussian window
        max_val: Maximum possible pixel value
    
    Returns:
        Average SSIM score
    """
    C1 = (0.01 * max_val) ** 2
    C2 = (0.03 * max_val) ** 2
    
    # Create Gaussian window
    kernel = torch.zeros(window_size)
    center = window_size // 2
    sigma = 1.5
    
    for i in range(window_size):
        x = i - center
        kernel[i] = math.exp(-(x**2) / (2 * sigma**2))
    
    kernel = kernel / kernel.sum()
    window = kernel.unsqueeze(0) * kernel.unsqueeze(1)
    window = window.unsqueeze(0).unsqueeze(0).to(clean.device)
    
    # Calculate means
    mu1 = F.conv2d(clean, window, padding=window_size//2)
    mu2 = F.conv2d(denoised, window, padding=window_size//2)
    
    mu1_sq = mu1 ** 2
    mu2_sq = mu2 ** 2
    mu1_mu2 = mu1 * mu2
    
    # Calculate variances and covariance
    sigma1_sq = F.conv2d(clean * clean, window, padding=window_size//2) - mu1_sq
    sigma2_sq = F.conv2d(denoised * denoised, window, padding=window_size//2) - mu2_sq
    sigma12 = F.conv2d(clean * denoised, window, padding=window_size//2) - mu1_mu2
    
    # SSIM formula
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / \
               ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    
    return ssim_map.mean().item()


print("Quality metrics implemented!")

## Evaluate All Methods

Let's compute PSNR and SSIM for all our denoising methods.

In [ ]:
# Get a larger test set for robust evaluation
test_noisy_list = []
test_clean_list = []

for noisy, clean in val_loader:
    test_noisy_list.append(noisy)
    test_clean_list.append(clean)
    if len(test_noisy_list) >= 10:  # Use 10 batches
        break

test_noisy_full = torch.cat(test_noisy_list).to(device)
test_clean_full = torch.cat(test_clean_list).to(device)

print(f"Evaluating on {len(test_noisy_full)} test samples...\n")

# Compute denoised versions
with torch.no_grad():
    # Traditional methods
    gaussian_blur_result = gaussian_blur_denoise(test_noisy_full, kernel_size=5, sigma=1.0)
    median_filter_result = median_filter_denoise(test_noisy_full, kernel_size=3)
    
    # Deep learning methods
    dae_result = dae(test_noisy_full)
    unet_result = unet(test_noisy_full)

# Calculate metrics
methods = {
    'Noisy (baseline)': test_noisy_full,
    'Gaussian Blur': gaussian_blur_result,
    'Median Filter': median_filter_result,
    'DAE': dae_result,
    'U-Net': unet_result,
}

results = []
for name, denoised in methods.items():
    psnr = calculate_psnr(test_clean_full, denoised)
    ssim = calculate_ssim(test_clean_full, denoised)
    results.append((name, psnr, ssim))
    print(f"{name:20s} - PSNR: {psnr:6.2f} dB, SSIM: {ssim:.4f}")

print("\nEvaluation complete!")

## Visualize Metrics Comparison

In [ ]:
# Extract data for plotting
method_names = [r[0] for r in results]
psnr_values = [r[1] for r in results]
ssim_values = [r[2] for r in results]

# Create bar plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Quantitative Comparison of Denoising Methods', fontsize=16, fontweight='bold')

# PSNR
colors = ['red', 'orange', 'orange', 'skyblue', 'green']
axes[0].bar(method_names, psnr_values, color=colors, alpha=0.7)
axes[0].set_ylabel('PSNR (dB)', fontweight='bold')
axes[0].set_title('Peak Signal-to-Noise Ratio (Higher is Better)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, v in enumerate(psnr_values):
    axes[0].text(i, v + 0.5, f'{v:.1f}', ha='center', va='bottom', fontweight='bold')

# SSIM
axes[1].bar(method_names, ssim_values, color=colors, alpha=0.7)
axes[1].set_ylabel('SSIM', fontweight='bold')
axes[1].set_title('Structural Similarity Index (Higher is Better)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylim([0, 1])
axes[1].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, v in enumerate(ssim_values):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nKey Findings:")
print(f"- U-Net achieves the highest PSNR ({psnr_values[-1]:.2f} dB)")
print(f"- U-Net achieves the highest SSIM ({ssim_values[-1]:.4f})")
print(f"- Deep learning methods outperform traditional methods significantly")

### 🤔 Reflection Questions

1. **Why do you think PSNR and SSIM rankings are similar but not identical?**
   - Hint: They measure different aspects of image quality

2. **Which metric better matches your visual perception?**
   - Look back at the visual results and compare

3. **When might high PSNR not guarantee good perceptual quality?**
   - Think about what MSE captures vs. what humans notice

---

# Part 7: Blind Denoising and Varying Noise Levels

So far, we've trained on a fixed noise level (σ=0.2 for Gaussian noise). But in practice:

1. **Noise levels vary** (different lighting conditions, cameras, etc.)
2. **Noise types may be unknown** (blind denoising)

## Theory: Generalization Challenges

### Problem 1: Varying Noise Levels

A model trained on σ=0.2 may fail on:
- Light noise (σ=0.1): Might oversmooth
- Heavy noise (σ=0.5): Might undersmooth

**Solution**: Train on **multiple noise levels** to learn a more robust mapping.

### Problem 2: Unknown Noise Type

Real-world images may have:
- Mix of noise types
- Unknown noise characteristics
- Non-Gaussian noise

**Solution**: Either:
1. Train separate models for each noise type (specialized)
2. Train a single model on mixed noise (generalist)
3. Use noise estimation techniques

Let's experiment with these scenarios!

## Experiment 1: Testing on Different Noise Levels

Let's see how our U-Net (trained on σ=0.2) performs on different noise levels.

In [ ]:
# Get clean test images
test_clean_samples = clean_images[:8].to(device)

# Test on different noise levels
noise_levels = [0.1, 0.2, 0.3, 0.4, 0.5]

fig, axes = plt.subplots(len(noise_levels), 8, figsize=(16, 10))
fig.suptitle('U-Net Performance on Varying Noise Levels (trained on σ=0.2)', 
             fontsize=16, fontweight='bold')

unet.eval()
with torch.no_grad():
    for i, noise_std in enumerate(noise_levels):
        # Add noise
        noisy = add_gaussian_noise(test_clean_samples, noise_std)
        
        # Denoise
        denoised = unet(noisy)
        
        # Calculate metrics
        psnr = calculate_psnr(test_clean_samples, denoised)
        ssim = calculate_ssim(test_clean_samples, denoised)
        
        # Visualize
        for j in range(8):
            axes[i, j].imshow(denoised[j, 0].cpu(), cmap='gray')
            axes[i, j].axis('off')
            if j == 0:
                axes[i, j].set_ylabel(f'σ={noise_std}\nPSNR={psnr:.1f}', 
                                     fontweight='bold', rotation=0, labelpad=60)

plt.tight_layout()
plt.show()

print("\nObservation: Model performs best near the training noise level (σ=0.2)")
print("Performance degrades for very light or very heavy noise.")

## Experiment 2: Quantitative Analysis Across Noise Levels

In [ ]:
# Comprehensive evaluation across noise levels
noise_levels_eval = np.arange(0.05, 0.55, 0.05)
psnr_results = []
ssim_results = []

test_batch = test_clean_full[:256]  # Use subset for speed

unet.eval()
with torch.no_grad():
    for noise_std in noise_levels_eval:
        noisy = add_gaussian_noise(test_batch, noise_std)
        denoised = unet(noisy)
        
        psnr = calculate_psnr(test_batch, denoised)
        ssim = calculate_ssim(test_batch, denoised)
        
        psnr_results.append(psnr)
        ssim_results.append(ssim)

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('U-Net Performance vs. Noise Level (trained on σ=0.2)', 
             fontsize=16, fontweight='bold')

axes[0].plot(noise_levels_eval, psnr_results, marker='o', linewidth=2)
axes[0].axvline(x=0.2, color='red', linestyle='--', alpha=0.5, label='Training noise')
axes[0].set_xlabel('Noise Standard Deviation (σ)', fontweight='bold')
axes[0].set_ylabel('PSNR (dB)', fontweight='bold')
axes[0].set_title('Peak Signal-to-Noise Ratio')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(noise_levels_eval, ssim_results, marker='s', linewidth=2, color='orange')
axes[1].axvline(x=0.2, color='red', linestyle='--', alpha=0.5, label='Training noise')
axes[1].set_xlabel('Noise Standard Deviation (σ)', fontweight='bold')
axes[1].set_ylabel('SSIM', fontweight='bold')
axes[1].set_title('Structural Similarity Index')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nKey Insight: Model performs well near training distribution,")
print("but degrades gracefully as noise level deviates from σ=0.2.")

## Experiment 3: Testing on Different Noise Types

Our U-Net was trained on **Gaussian noise**. Let's test it on **salt-and-pepper** and **speckle** noise!

In [ ]:
# Get test samples
test_clean_samples = clean_images[:4].to(device)

# Create different noise types
noisy_gaussian = add_gaussian_noise(test_clean_samples, 0.2)
noisy_salt_pepper = add_salt_pepper_noise(test_clean_samples, 0.05, 0.05)
noisy_speckle = add_speckle_noise(test_clean_samples, 0.3)

# Denoise with U-Net
unet.eval()
with torch.no_grad():
    denoised_gaussian = unet(noisy_gaussian)
    denoised_salt_pepper = unet(noisy_salt_pepper)
    denoised_speckle = unet(noisy_speckle)

# Calculate metrics
psnr_g = calculate_psnr(test_clean_samples, denoised_gaussian)
psnr_sp = calculate_psnr(test_clean_samples, denoised_salt_pepper)
psnr_sk = calculate_psnr(test_clean_samples, denoised_speckle)

ssim_g = calculate_ssim(test_clean_samples, denoised_gaussian)
ssim_sp = calculate_ssim(test_clean_samples, denoised_salt_pepper)
ssim_sk = calculate_ssim(test_clean_samples, denoised_speckle)

# Visualize
fig, axes = plt.subplots(3, 8, figsize=(16, 6))
fig.suptitle('U-Net on Different Noise Types (trained on Gaussian)', 
             fontsize=16, fontweight='bold')

for i in range(4):
    # Gaussian noise
    axes[0, i].imshow(noisy_gaussian[i, 0].cpu(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Gaussian\nNoisy', fontweight='bold', rotation=0, labelpad=40)
    
    axes[0, i+4].imshow(denoised_gaussian[i, 0].cpu(), cmap='gray')
    axes[0, i+4].axis('off')
    if i == 0:
        axes[0, i+4].set_ylabel(f'Denoised\nPSNR={psnr_g:.1f}', 
                               fontweight='bold', rotation=0, labelpad=60)
    
    # Salt-and-pepper noise
    axes[1, i].imshow(noisy_salt_pepper[i, 0].cpu(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Salt-Pepper\nNoisy', fontweight='bold', rotation=0, labelpad=40)
    
    axes[1, i+4].imshow(denoised_salt_pepper[i, 0].cpu(), cmap='gray')
    axes[1, i+4].axis('off')
    if i == 0:
        axes[1, i+4].set_ylabel(f'Denoised\nPSNR={psnr_sp:.1f}', 
                               fontweight='bold', rotation=0, labelpad=60)
    
    # Speckle noise
    axes[2, i].imshow(noisy_speckle[i, 0].cpu(), cmap='gray')
    axes[2, i].axis('off')
    if i == 0:
        axes[2, i].set_ylabel('Speckle\nNoisy', fontweight='bold', rotation=0, labelpad=40)
    
    axes[2, i+4].imshow(denoised_speckle[i, 0].cpu(), cmap='gray')
    axes[2, i+4].axis('off')
    if i == 0:
        axes[2, i+4].set_ylabel(f'Denoised\nPSNR={psnr_sk:.1f}', 
                               fontweight='bold', rotation=0, labelpad=60)

plt.tight_layout()
plt.show()

print("\nPerformance on Different Noise Types:")
print(f"Gaussian (trained):    PSNR={psnr_g:.2f} dB, SSIM={ssim_g:.4f}")
print(f"Salt-Pepper (unseen): PSNR={psnr_sp:.2f} dB, SSIM={ssim_sp:.4f}")
print(f"Speckle (unseen):     PSNR={psnr_sk:.2f} dB, SSIM={ssim_sk:.4f}")
print("\nConclusion: Model generalizes reasonably to speckle, but struggles with salt-and-pepper!")

### 🤔 Reflection Questions

1. **Why does the model struggle more with salt-and-pepper than speckle noise?**
   - Hint: Consider how similar each noise type is to Gaussian noise

2. **How could we make a model that works well on all noise types?**
   - Think about training data diversity

3. **What are the trade-offs between specialist vs. generalist models?**
   - Consider performance, training cost, and deployment complexity

---

# Part 8: Advanced Topics and Future Directions

## Real-World Considerations

### 1. Noise Estimation

In practice, we often don't know the noise level or type. **Noise estimation** techniques can help:

- **Blind noise estimation**: Estimate noise parameters from the image itself
- **Patch-based methods**: Analyze homogeneous regions
- **Learning-based**: Train a network to predict noise parameters

### 2. Real vs. Synthetic Noise

Synthetic noise (Gaussian, salt-and-pepper) is a **simplification**. Real-world noise:

- Is often **signal-dependent** (Poisson noise in low light)
- May be **spatially correlated**
- Can have **camera-specific patterns**
- Includes **compression artifacts**

**Solution**: Use real noisy-clean image pairs (e.g., high ISO + low ISO) for training.

### 3. Perceptual Loss Functions

MSE loss doesn't always match human perception. Alternatives:

- **Perceptual loss**: Compare features from pre-trained networks (VGG)
- **Adversarial loss**: Use a discriminator to encourage realistic outputs
- **SSIM loss**: Directly optimize structural similarity

$$\mathcal{L} = \alpha \cdot \mathcal{L}_{MSE} + \beta \cdot \mathcal{L}_{perceptual} + \gamma \cdot \mathcal{L}_{adversarial}$$

### 4. Computational Efficiency

For real-time applications (video, mobile), we need:

- **Lighter architectures**: MobileNet-style designs, depthwise convolutions
- **Quantization**: Reduce precision (float32 → int8)
- **Pruning**: Remove unnecessary parameters
- **Knowledge distillation**: Train small model to mimic large one

## Modern Approaches

Recent advances in denoising:

### 1. Self-Supervised Denoising

Train without clean images!

- **Noise2Noise**: Train on pairs of noisy images of the same scene
- **Noise2Void**: Use neighboring pixels as targets
- **Noise2Self**: Split noisy image into subsets

### 2. Transformer-Based Denoisers

Apply attention mechanisms:

- **Swin Transformer**: Hierarchical vision transformers for image restoration
- **Restormer**: Efficient transformer for high-resolution denoising
- Better at capturing long-range dependencies

### 3. Diffusion Models

State-of-the-art generative approach:

- Iteratively refine noisy image through learned denoising steps
- Excellent perceptual quality
- Computationally expensive

### 4. Blind Denoising

Handle unknown noise:

- **Conditional models**: Network takes noise parameters as input
- **Noise disentanglement**: Separate noise estimation and removal
- **Adaptive methods**: Adjust to local noise characteristics

## Suggested Experiments

Here are some ideas for further exploration:

### Experiment 1: Multi-Noise Training
```python
# Train U-Net on mixture of noise types
class MultiNoiseDataset(Dataset):
    def __getitem__(self, idx):
        clean = self.clean_images[idx]
        noise_type = random.choice(['gaussian', 'salt_pepper', 'speckle'])
        noisy = add_noise(clean, noise_type)  # Random noise each time
        return noisy, clean
```

**Question**: Does this improve generalization across noise types?

### Experiment 2: Color Image Denoising
```python
# Use CIFAR-10 (color images) instead of MNIST
unet_rgb = UNet(in_channels=3)  # RGB channels
```

**Question**: How do results differ for color vs. grayscale?

### Experiment 3: Deeper U-Net
```python
# Add more encoder/decoder levels
# 28x28 → 14x14 → 7x7 → 3x3 (deeper bottleneck)
```

**Question**: Does deeper architecture improve quality? What's the trade-off?

### Experiment 4: Residual Learning
```python
# Instead of predicting clean image, predict noise
def forward(self, x):
    noise_pred = self.unet(x)
    clean = x - noise_pred  # Subtract predicted noise
    return clean
```

**Question**: Does learning the residual (noise) converge faster?

### Experiment 5: Perceptual Loss
```python
# Use VGG features for perceptual loss
from torchvision.models import vgg16
vgg = vgg16(pretrained=True).features[:16]  # Use first layers

loss = mse_loss + lambda * perceptual_loss(vgg(denoised), vgg(clean))
```

**Question**: Does this improve perceptual quality?

---

# Summary and Key Takeaways

## What We Learned

### 1. Image Noise Fundamentals
- **Gaussian noise**: Most common, additive, smooth
- **Salt-and-pepper**: Sparse impulse noise, high contrast
- **Speckle**: Multiplicative, granular texture
- Different noise types require different approaches

### 2. Traditional Denoising Methods
- **Gaussian blur**: Simple but blurs edges
- **Median filter**: Excellent for salt-and-pepper, preserves edges
- Make assumptions about noise and image structure
- Limited adaptability

### 3. Denoising Autoencoder (DAE)
- Learns to map noisy → clean through neural networks
- Encoder-decoder architecture with bottleneck
- **Limitation**: Information loss at bottleneck
- Still effective, significantly better than traditional methods

### 4. U-Net Architecture
- **Skip connections** preserve fine details
- Multi-scale feature fusion
- **State-of-the-art** for image-to-image tasks
- Significantly outperforms simple autoencoders

### 5. Evaluation Metrics
- **PSNR**: Signal-to-noise ratio in dB, simple but limited
- **SSIM**: Structural similarity, better matches perception
- Quantitative metrics complement visual inspection

### 6. Generalization Challenges
- Models trained on specific noise struggle on others
- Performance degrades away from training distribution
- **Blind denoising** (unknown noise) is challenging
- Trade-off between specialist and generalist models

## Key Insights

1. **Skip connections are crucial** for preserving spatial details in image reconstruction tasks

2. **Training data diversity** directly impacts generalization to different noise types and levels

3. **Deep learning outperforms traditional methods** significantly, especially for complex noise patterns

4. **No single metric captures all aspects** of image quality - use multiple metrics and visual inspection

5. **Real-world denoising** is more complex than synthetic noise - domain gap is important

## Next Steps

To deepen your understanding:

1. **Try the suggested experiments** above
2. **Read papers** on Noise2Noise, Restormer, and diffusion-based denoising
3. **Implement residual learning** for U-Net
4. **Explore real-world datasets** like SIDD or DND
5. **Experiment with color images** (CIFAR-10, ImageNet)

## Congratulations!

You've completed a comprehensive journey through image denoising, from fundamental concepts to state-of-the-art architectures. You now understand:

- The mathematics of noise and denoising
- Traditional and modern approaches
- Architecture design principles (autoencoders, U-Net, skip connections)
- Evaluation methodologies
- Real-world challenges and solutions

This knowledge forms a strong foundation for tackling other image restoration tasks like super-resolution, inpainting, and artifact removal!

---

## References and Further Reading

### Foundational Papers

1. **Vincent et al. (2008)**: "Extracting and Composing Robust Features with Denoising Autoencoders"
   - Original denoising autoencoder paper

2. **Ronneberger et al. (2015)**: "U-Net: Convolutional Networks for Biomedical Image Segmentation"
   - Introduced U-Net architecture

3. **Zhang et al. (2017)**: "Beyond a Gaussian Denoiser: Residual Learning of Deep CNN for Image Denoising"
   - DnCNN, residual learning for denoising

### Modern Approaches

4. **Lehtinen et al. (2018)**: "Noise2Noise: Learning Image Restoration without Clean Data"
   - Self-supervised denoising

5. **Zamir et al. (2022)**: "Restormer: Efficient Transformer for High-Resolution Image Restoration"
   - Transformer-based denoising

6. **Ho et al. (2020)**: "Denoising Diffusion Probabilistic Models"
   - Diffusion models for generation and restoration

### Datasets

- **MNIST**: Handwritten digits (used in this notebook)
- **CIFAR-10**: Color natural images
- **SIDD**: Smartphone Image Denoising Dataset (real noise)
- **DND**: Darmstadt Noise Dataset (real noise)
- **BSD68**: Berkeley Segmentation Dataset (benchmark)

### Tools and Libraries

- **PyTorch**: Deep learning framework
- **scikit-image**: Image processing in Python
- **OpenCV**: Computer vision library
- **MATLAB Image Processing Toolbox**: Traditional methods

Happy denoising! 🚀